# Sqoop jobs → intermediate YAML
Runs `agentic.sqoop_yaml` (the same converter behind `POST /api/sqoop-yaml` and the *Sqoop jobs to YAML* tab of
`ui/compare_app.py`). Keep this notebook in `sqoop_poc/`; it works from the project root. Put your Sqoop commands in
`SQOOP_FILE`, one per line.

In [ ]:
import os, sys, subprocess
import yaml

ROOT = os.path.abspath("..")                      # the project folder
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "trino"])

from agentic.spark_compare import load_source
from agentic.sqoop_yaml import Options, convert

SQOOP_FILE = os.path.join(ROOT, "samples", "sqoop_jobs.txt")   # your Sqoop commands
DDL_FILE   = "01_schema.sql"         # source table definitions (or set TRINO_URL instead)
TRINO_URL  = None                    # e.g. "http://172.171.240.8:8082" (reads the landed copies)
TRINO_SCHEMA = "hive.sqoop_poc_spark"  # where Trino finds those tables
OUT_DIR    = os.path.join(ROOT, "output")

TARGET = Options(project_id="sqoop_poc", target_catalog="ice", target_namespace="sqoop_poc_iceberg",
                 target_storage="hdfs://172.16.0.4:9000/user/hive/warehouse/sqoop_poc_iceberg.db")

## 1. Generate one YAML file per Sqoop job

In [ ]:
ddl = load_source(path=DDL_FILE) if not TRINO_URL and os.path.exists(DDL_FILE) else None
res = convert(load_source(path=SQOOP_FILE), ddl, TARGET,
              trino_url=TRINO_URL, trino_schema=TRINO_SCHEMA if TRINO_URL else None, out_base=OUT_DIR)
for w in res["report"]["warnings"]:
    print("note:", w)
for j in res["report"]["jobs"]:
    print("%-28s %-26s findings=%d (blocking execution: %d)" % (j["job_id"], j["load_mode"], j["findings"], j["blocking_execution"]))
print("files in", res["folder"])

## 2. Look at one file

In [ ]:
files = res["files"]["yaml"]
assert files, "no YAML generated: check the output of the cell above"
print("\n".join(sorted(files.values())))
print(res["yaml"][sorted(files)[0]])

## 3. Summary of findings across all jobs

In [ ]:
rows = res["report"]["findings"]
try:
    import pandas as pd
    pd.set_option("display.max_colwidth", 120)
    display(pd.DataFrame(rows)[["job", "id", "category", "severity", "blocks", "evidence"]])
except ImportError:
    for r_ in rows: print(r_)

## 4. Coverage: how every Sqoop option was handled

In [ ]:
for f in sorted(files.values()):
    spec = yaml.safe_load(open(f))
    print(spec["job_id"])
    for c in spec["coverage"]:
        print("   %-28s %-12s %s" % (c["construct"], c["status"], (c.get("refs") or c.get("finding_refs") or [c.get("reason", "")])[0]))